# TP 8 — Mesurer l'équité et son arbitrage

> Support théorique : [cours_ethique_securite_regulation.md](cours_ethique_securite_regulation.md).

**Objectif.** Rendre tangibles les notions du module 8 : mesurer des métriques **par sous-groupe**
(taux de sélection, TPR, FPR), constater qu'une **moyenne globale masque une disparité**, et vérifier
« sur pièces » qu'on ne peut **pas** satisfaire simultanément plusieurs définitions d'équité quand les
**taux de base diffèrent** (module 8, §2.2).

Données synthétiques, hors ligne. L'attribut sensible est noté `A` (deux groupes). On n'utilise **pas** `A`
comme feature du modèle — mais on verra que cela ne suffit pas à garantir l'équité (module 8, §1).

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(0)
pd.set_option("display.width", 100)
print("Prêt.")

Prêt.


## 1. Données avec des taux de base différents

On simule deux groupes dont la **prévalence** de l'issue positive diffère fortement
(groupe 0 : ~20 % ; groupe 1 : ~50 %). Deux features `x1, x2` sont prédictives de l'issue `y`.
Cette différence de taux de base est fréquente dans le monde réel et suffit à créer des tensions d'équité.

In [2]:
n = 4000
A = rng.integers(0, 2, size=n)                       # attribut sensible : groupe 0 ou 1
base = np.where(A == 0, 0.20, 0.50)                  # taux de base différents par groupe
y = (rng.random(n) < base).astype(int)

x1 = y + rng.normal(0, 1.0, size=n)                  # features prédictives de y
x2 = 0.5 * y + rng.normal(0, 1.0, size=n)
X = np.column_stack([x1, x2])

print("Taux de base observés :")
print(f"  groupe 0 : {y[A==0].mean():.3f}")
print(f"  groupe 1 : {y[A==1].mean():.3f}")
assert y[A == 1].mean() > y[A == 0].mean() + 0.15, "les taux de base doivent nettement différer" 

Taux de base observés :
  groupe 0 : 0.204
  groupe 1 : 0.506


In [3]:
X_tr, X_te, y_tr, y_te, A_tr, A_te = train_test_split(
    X, y, A, test_size=0.4, random_state=0, stratify=y
)
# Le modèle n'utilise PAS l'attribut sensible A comme feature.
clf = LogisticRegression(max_iter=1000, random_state=0).fit(X_tr, y_tr)
proba = clf.predict_proba(X_te)[:, 1]
print("Accuracy globale :", round((clf.predict(X_te) == y_te).mean(), 3))

Accuracy globale : 0.728


## 2. Métriques par sous-groupe

On définit, pour un seuil donné, le **taux de sélection** (part de prédictions positives — parité
démographique), le **TPR** (rappel — égalité des chances) et le **FPR**. On les calcule **par groupe**
(module 8, §2.3).

In [4]:
def taux(proba, y_true, seuil):
    pred = (proba >= seuil).astype(int)
    sel = pred.mean()
    tpr = pred[y_true == 1].mean() if (y_true == 1).any() else float("nan")
    fpr = pred[y_true == 0].mean() if (y_true == 0).any() else float("nan")
    return sel, tpr, fpr

def tableau(proba, y_true, A, seuils):
    lignes = []
    for g in (0, 1):
        m = A == g
        s = seuils[g] if isinstance(seuils, dict) else seuils
        sel, tpr, fpr = taux(proba[m], y_true[m], s)
        lignes.append({"groupe": g, "seuil": round(s, 3),
                       "taux_selection": round(sel, 3),
                       "TPR": round(tpr, 3), "FPR": round(fpr, 3)})
    return pd.DataFrame(lignes)

seuil_global = 0.5
tab = tableau(proba, y_te, A_te, seuil_global)
print(f"Seuil commun = {seuil_global}")
print(tab.to_string(index=False))

sr0, sr1 = tab.loc[0, "taux_selection"], tab.loc[1, "taux_selection"]
print(f"\nÉcart de taux de sélection entre groupes : {abs(sr0 - sr1):.3f}")
assert abs(sr0 - sr1) > 0.05, "à seuil commun, les taux de sélection diffèrent (pas de parité démographique)"
print("-> Une accuracy globale correcte cache une disparité de traitement entre groupes.")

Seuil commun = 0.5
 groupe  seuil  taux_selection   TPR   FPR
      0    0.5           0.208 0.487 0.141
      1    0.5           0.343 0.512 0.162

Écart de taux de sélection entre groupes : 0.135
-> Une accuracy globale correcte cache une disparité de traitement entre groupes.


## 3. L'arbitrage : égaliser un critère en déséquilibre un autre

Forçons la **parité démographique** en choisissant un **seuil par groupe** qui égalise le taux de
sélection (au taux global). On observe alors que le **TPR** (égalité des chances) diffère entre groupes :
avec des taux de base différents, on ne peut satisfaire les deux à la fois (module 8, §2.2).

In [5]:
cible = (proba >= 0.5).mean()          # taux de sélection global visé
seuils_parite = {}
for g in (0, 1):
    scores_g = proba[A_te == g]
    # seuil tel que la part de scores >= seuil vaut approximativement 'cible'
    seuils_parite[g] = float(np.quantile(scores_g, 1 - cible))

tab2 = tableau(proba, y_te, A_te, seuils_parite)
print("Seuils choisis pour égaliser le taux de sélection :")
print(tab2.to_string(index=False))

ecart_sel = abs(tab2.loc[0, "taux_selection"] - tab2.loc[1, "taux_selection"])
ecart_tpr = abs(tab2.loc[0, "TPR"] - tab2.loc[1, "TPR"])
print(f"\nAprès égalisation : écart de sélection = {ecart_sel:.3f} | écart de TPR = {ecart_tpr:.3f}")
assert ecart_sel < 0.05, "les taux de sélection sont désormais quasi égaux (parité démographique)"
assert ecart_tpr > 0.05, "...mais le TPR diffère : on ne peut satisfaire les deux critères à la fois"
print("-> Choisir une définition d'équité est un ARBITRAGE de valeurs, pas un réglage neutre.")

Seuils choisis pour égaliser le taux de sélection :
 groupe  seuil  taux_selection   TPR   FPR
      0  0.437           0.276 0.558 0.208
      1  0.558           0.276 0.433 0.108

Après égalisation : écart de sélection = 0.000 | écart de TPR = 0.125
-> Choisir une définition d'équité est un ARBITRAGE de valeurs, pas un réglage neutre.


## 4. Retirer l'attribut sensible ne suffit pas

Le modèle n'a jamais vu `A`. Pourtant les disparités existent, car les features `x1, x2` sont corrélées à
l'issue dont le taux de base dépend du groupe : elles agissent comme **proxys** (module 8, §1). L'équité
se traite donc par la mesure, l'arbitrage documenté et, si besoin, une atténuation — pas par la simple
omission de la variable sensible.

In [6]:
# Vérifions que x1 porte de l'information corrélée au groupe (via y), donc un proxy possible.
corr_x1_A = np.corrcoef(X_te[:, 0], A_te)[0, 1]
print(f"Corrélation feature x1 / groupe A : {corr_x1_A:.3f}")
assert abs(corr_x1_A) > 0.05, "une feature 'neutre' encode indirectement le groupe (proxy)" 

Corrélation feature x1 / groupe A : 0.152


## 5. Exercice guidé — métriques par groupe

Écrivez `metriques_groupe(y_true, y_pred, A)` qui renvoie, pour chaque groupe, un dict
`{"selection": ..., "TPR": ...}`. **Critère de réussite** : correspond aux valeurs attendues sur le petit
exemple ci-dessous.

In [7]:
def metriques_groupe(y_true, y_pred, A):
    # TODO : pour chaque groupe g présent dans A, calculer
    #   selection = moyenne de y_pred dans le groupe
    #   TPR       = moyenne de y_pred parmi les y_true == 1 du groupe
    # Renvoyer un dict {g: {"selection": ..., "TPR": ...}}
    raise NotImplementedError("À compléter")

# y_true = np.array([1, 0, 1, 1, 0, 1])
# y_pred = np.array([1, 0, 0, 1, 1, 1])
# A      = np.array([0, 0, 0, 1, 1, 1])
# res = metriques_groupe(y_true, y_pred, A)
# assert np.isclose(res[0]["TPR"], 0.5) and np.isclose(res[1]["selection"], 1.0)

### Solution

In [8]:
def metriques_groupe(y_true, y_pred, A):
    y_true, y_pred, A = map(np.asarray, (y_true, y_pred, A))
    out = {}
    for g in np.unique(A):
        m = A == g
        pos = m & (y_true == 1)
        out[int(g)] = {
            "selection": float(y_pred[m].mean()),
            "TPR": float(y_pred[pos].mean()) if pos.any() else float("nan"),
        }
    return out

y_true = np.array([1, 0, 1, 1, 0, 1])
y_pred = np.array([1, 0, 0, 1, 1, 1])
A_ex   = np.array([0, 0, 0, 1, 1, 1])
res = metriques_groupe(y_true, y_pred, A_ex)
print(res)
assert np.isclose(res[0]["TPR"], 0.5), "groupe 0 : 1 positif prédit sur 2 réels"
assert np.isclose(res[1]["selection"], 1.0), "groupe 1 : tous prédits positifs"
print("OK : métriques par groupe correctes.")

{0: {'selection': 0.3333333333333333, 'TPR': 0.5}, 1: {'selection': 1.0, 'TPR': 1.0}}
OK : métriques par groupe correctes.


## Récapitulatif

- **Décomposez** toujours les métriques par sous-groupe : une moyenne globale masque les disparités.
- Parité démographique, égalité des chances et calibration sont **des définitions distinctes** ; avec des
  taux de base différents, on ne peut généralement pas toutes les satisfaire (résultat d'impossibilité).
- Choisir un critère d'équité est un **arbitrage de valeurs** à documenter, avec les personnes affectées et
  une voie de recours (module 8, §2 et §8).
- **Retirer l'attribut sensible ne suffit pas** : des proxys subsistent.

Ce TP mesure et arbitre ; il ne « résout » pas l'équité. Retour au [README](../README.md) et au
[glossaire](../GLOSSAIRE.md) pour l'ensemble du cursus.